# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut

  Using cached getorg-0.3.1-py2.py3-none-any.whl.metadata (326 bytes)
  Using cached geographiclib-2.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached pynacl-1.6.2-cp38-abi3-win_amd64.whl.metadata (10 kB)
Using cached getorg-0.3.1-py2.py3-none-any.whl (39 kB)
Using cached geographiclib-2.1-py3-none-any.whl (40 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 598.5 kB/s eta 0:00:06
   ----- ---------------------------------- 0.5/3.8 MB 598.5 kB/s eta 0:00:06
   ----- ---------------------------------- 0.5/3.8 MB 598.5 kB/s eta 0:00:06
   -------- ------------------------------- 0.8/3.8 MB 558.9 kB/s eta 0:00:06
   -------- ------------------------------- 0.8/3.8 MB 558.9 kB/s eta 0:00:06
   ---------- ----------------------------- 1.0/3.8 MB 572.0 kB/s eta 0:00:05
   ----

In [ ]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    print(file)
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)